# CivicPulse - CPU vs GPU Benchmark (Track B)

**What this proves:** the ward officer changes a filter (category / time window /
severity) and CivicPulse must *re-aggregate the entire multi-year complaint corpus*
across all 198 wards to re-rank hotspots. On CPU this stalls; on an NVIDIA GPU with
cuDF it is instant - so the analysis stays **interactive** instead of overnight-batch.

> **Before you run:** menu -> Runtime -> Change runtime type -> **GPU** (T4 is fine).
> Then Runtime -> **Run all**.


## 1. Confirm a GPU is attached

In [ ]:
!nvidia-smi

## 2. Install cuDF (NVIDIA RAPIDS)
Skips instantly if the runtime already has it.

In [ ]:
try:
    import cudf  # noqa
    print('cudf already available:', cudf.__version__)
except Exception:
    !pip install -q --extra-index-url=https://pypi.nvidia.com cudf-cu12
    import cudf
    print('installed cudf:', cudf.__version__)

## 3. Load the real BBMP data + scale it to city-history size
We pull the real CivicPulse pipeline from GitHub, load real Bengaluru grievances,
then amplify to a realistic multi-year, all-wards volume for the benchmark.

In [ ]:
import sys, time
import numpy as np
import pandas as pd

!git clone -q https://github.com/reply2vikas/civicpulse.git || echo 'already cloned'
sys.path.insert(0, '/content/civicpulse')

from src.pipeline import load_grievances
from src.severity import CATEGORY_SEVERITY, DEFAULT_SEVERITY

base = load_grievances()          # real 2025 grievances (falls back to sample offline)
print('real rows loaded:', len(base))

In [ ]:
# Fast, vectorised amplification (no per-row Python) so setup stays quick.
def make_big(base, n, seed=7):
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(base), n)
    b = base.iloc[idx].reset_index(drop=True)
    cat = (b['category'].astype(str)
             .str.replace(r'\s+', ' ', regex=True).str.strip())
    status = rng.choice(['Registered','In Progress','Closed','ReOpen'],
                        size=n, p=[0.45,0.15,0.33,0.07])
    out = pd.DataFrame({
        'ward': b['ward'].astype(str).values,
        'category': cat.values,
        'complaint_id': np.arange(n, dtype='int64'),
        'is_open_i': (status != 'Closed').astype('int8'),
    })
    out['severity'] = (out['category'].map(CATEGORY_SEVERITY)
                       .fillna(DEFAULT_SEVERITY).astype('int8'))
    return out

N_ROWS = 3_000_000   # ~ multi-year, all-ward city scale. Bump to 6-10M if you like.
big = make_big(base, N_ROWS)
print(f'benchmark frame: {len(big):,} rows, {big.ward.nunique()} wards, '
      f'{big.memory_usage(deep=True).sum()/1e6:.0f} MB')
big.head()

## 4. The hot path: re-aggregate every ward x category
This is the exact compute that runs on **every filter change**. Same function runs
on a pandas frame (CPU) and a cuDF frame (GPU).

In [ ]:
def heavy_aggregate(df):
    return (df.groupby(['ward', 'category'])
              .agg({'complaint_id': 'count',
                    'is_open_i': 'mean',
                    'severity': 'max'}))

def timeit(fn, *a, repeat=5):
    best = float('inf')
    for _ in range(repeat):
        t0 = time.perf_counter(); fn(*a); best = min(best, time.perf_counter()-t0)
    return best

## 5. CPU (pandas)

In [ ]:
cpu_s = timeit(heavy_aggregate, big)
print(f'CPU  (pandas): {cpu_s*1000:8.1f} ms')
heavy_aggregate(big).head()

## 6. GPU (cuDF)

In [ ]:
t0 = time.perf_counter()
gbig = cudf.from_pandas(big)
convert_s = time.perf_counter() - t0

gpu_s = timeit(heavy_aggregate, gbig)
print(f'host->GPU copy : {convert_s*1000:8.1f} ms  (one-time on load)')
print(f'GPU  (cuDF)    : {gpu_s*1000:8.1f} ms')
print(f'\nSpeedup on the re-aggregation: {cpu_s/gpu_s:5.1f}x faster')

## 7. The result

In [ ]:
import matplotlib.pyplot as plt

labels = ['CPU\n(pandas)', 'GPU\n(cuDF)']
vals = [cpu_s*1000, gpu_s*1000]
fig, ax = plt.subplots(figsize=(5,4))
bars = ax.bar(labels, vals, color=['#9aa0a6', '#76b900'])  # NVIDIA green
ax.set_ylabel('re-aggregation time (ms, lower is better)')
ax.set_title(f'CivicPulse: {N_ROWS:,} complaints re-ranked\n'
             f'{cpu_s/gpu_s:.0f}x faster on GPU')
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v, f'{v:.0f} ms',
            ha='center', va='bottom')
plt.tight_layout(); plt.show()

print(f'''
SO WHAT:
  At {N_ROWS:,} complaints, re-ranking every ward x category takes
  {cpu_s*1000:.0f} ms on CPU vs {gpu_s*1000:.0f} ms on GPU ({cpu_s/gpu_s:.0f}x).
  A ward officer can slide the date/severity filters and see the city
  re-prioritise INSTANTLY - interactive decision-making instead of an
  overnight batch job.''')